In [1]:
#| default_exp pipeline

In [2]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

A pipeline is a plan, a record of how each step went, and one pty the steps run on.

Every step is a child process on a pseudo-terminal. A step that asks for a password can be answered with `write`, and a step that draws a progress bar draws it. The plan is a file. The results and the transcript are on the object and go when it does.

Nothing here imports the cloud half of pullup. A pipeline that runs `pytest` needs fastcore and ptymini.

In [3]:
#| export
from __future__ import annotations
import asyncio, json, os, shlex, time
from shutil import which
from fastcore.all import L, Path, first
from fastcore.ansi import strip_ansi

from pullup.env import EnvStore, venv_env
from pullup.project import Step, default_steps, release_flow, app_project

In [4]:
#| export
#: Bytes of transcript one pipeline keeps. A viewer that joins late is primed with this much.
SCROLLBACK = 200_000
#: Where a pipeline's plan is written, relative to the project. A caller may name another.
DIR = '.pullup'

class PipelineError(RuntimeError): pass

`PipelineError` is the only exception this module raises. A step that exits non-zero does not raise. A failure is a result: the step is `failed`, its exit code is recorded, and `state` reports both.

In [5]:
#| export
def resolve_script(argv, python=None, env=None):
    "The executable to spawn, accepting either spelling of a hyphenated command."
    name = argv[0]
    spellings = (name, name.replace('_', '-'), name.replace('-', '_'))
    bindir = str(Path(python).parent) if python else ''
    for path in (bindir, (env or os.environ).get('PATH')):
        if not path: continue
        for candidate in spellings:
            if hit := which(candidate, path=path): return [hit, *argv[1:]]
    return argv

def uv_argv(root, argv):
    """`argv` through `uv run` where uv owns the project, else None.

    uv syncs the lock before it runs anything, so a venv that has drifted is repaired rather than
    failing halfway through with an import error.
    """
    root = Path(root)
    if not (root/'uv.lock').exists(): return None
    exe = which('uv')
    if not exe: return None
    # Spelled the way the project spells it. uv runs the project's own script where there is one
    # and falls through to PATH where there is not, and PATH is where another interpreter's
    # differently-spelled copy of the same tool is waiting.
    name = argv[0]
    bindir = root/'.venv'/'bin'
    hit = first(n for n in (name, name.replace('_', '-'), name.replace('-', '_'))
                if (bindir/n).exists())
    return [exe, 'run', '--project', str(root), hit or name.replace('_', '-'), *argv[1:]]

`resolve_script` answers with the absolute path to the executable. Where nothing resolves it answers with `argv` unchanged, and the spawn fails naming the command the plan asked for.

It looks in the interpreter's own directory before `PATH`, so a project's own `pytest` wins over the one the shell would find. At each place it tries three spellings: the name as written, underscores hyphenated, and hyphens underscored. nbdev installs `nbdev_export` and a plan may spell it `nbdev-export`.

`uv_argv` answers with None unless the project has a `uv.lock` and `uv` is on `PATH`. Where it answers, the command runs under `uv run --project`, which syncs the lock first, so a virtual environment that has drifted is repaired rather than failing halfway through with an import error.

The command is spelled the way the project's own `.venv/bin` spells it. uv falls through to `PATH` for a name it does not find in the project, and `PATH` is where another interpreter's copy of the same tool is waiting.

In [6]:
#| hide
#: A hyphen-spelled tool and a uv on PATH, so the examples do not depend on what is installed here.
tmp = TemporaryDirectory(); root = Path(tmp.name)
bindir = root/'.venv'/'bin'; bindir.mkdir(parents=True)
for n in ('my_tool', 'uv'): (bindir/n).write_text('#!/bin/sh\n'); (bindir/n).chmod(0o755)
os.environ['PATH'] = str(bindir) + os.pathsep + os.environ['PATH']

In [7]:
argv = resolve_script(['my-tool', '--flag'], python=bindir/'python')
Path(argv[0]).name, argv[1:], resolve_script(['not-installed-anywhere'])

('my_tool', ['--flag'], ['not-installed-anywhere'])

In [8]:
(root/'uv.lock').write_text('')
argv = uv_argv(root, ['my-tool', '-q'])
argv[0], argv[3] = Path(argv[0]).name, '<project>'
argv

['uv', 'run', '--project', '<project>', 'my_tool', '-q']

In [9]:
#| hide
test_eq(uv_argv(root/'nothing', ['pytest']), None)   # no lock, so uv owns nothing here
test_eq(resolve_script(['not-installed-anywhere']), ['not-installed-anywhere'])

In [10]:
#| export
class Pipeline:
    "A named sequence of commands for one project: its steps, its state, and its terminal."
    FILE = 'pipeline.json'
    KIND = 'pipeline'
    @staticmethod
    def defaults(root): return []
    def __init__(self,
        root,             # the project the commands run in
        python=None,      # the interpreter whose environment they run in; None uses this process's
        env=None,         # an `EnvStore`, or anything with `.get(key)` and `.values(keys)`
        dir=DIR,          # where the plan is written, relative to `root`
    ):
        self.root = Path(root).expanduser().resolve()
        self.python, self.dir = python, dir
        self.env = env if env is not None else EnvStore()
        self.steps = self._load()
        self.results, self.active = {}, ''
        self.pty = self._pump = None
        self._subs, self._auto = set(), True
        self.log = bytearray()
    @property
    def path(self): return self.root/self.dir/self.FILE
    def _load(self):
        try: raw = json.loads(self.path.read_text(encoding='utf-8'))
        except (OSError, ValueError): return self.defaults(self.root)
        steps = raw.get('steps') if isinstance(raw, dict) else raw
        rows = [r for r in (steps or ()) if isinstance(r, dict) and r.get('cmd')]
        out = [Step(id=str(row.get('id') or f'step{i + 1}'),
            label=str(row.get('label') or row.get('id') or 'step'),
            cmd=str(row['cmd']), doc=str(row.get('doc') or ''),
            needs=[str(x) for x in (row.get('needs') or ())],
            argv=[str(x) for x in (row.get('argv') or ())],
            meta=dict(row.get('meta') or {})) for i, row in enumerate(rows)]
        return out or self.defaults(self.root)
    def save(self, steps):
        "Persist an edited pipeline. Refuses a plan with no runnable command in it."
        rows = [s for s in (steps or ()) if str((s or {}).get('cmd') or '').strip()]
        if not rows: raise PipelineError('a pipeline needs at least one step with a command')
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps({'steps': rows}, indent=2) + '\n', encoding='utf-8')
        self.steps = self._load()
        return self.state()
    def reset_plan(self):
        "Back to the default pipeline for this kind of project, without deleting the file."
        self.steps = self.defaults(self.root)
        return self.state()
    def step(self, step_id):
        found = first(self.steps, lambda s: s.id == str(step_id))
        if found is None: raise PipelineError(f'unknown {self.KIND} step: {step_id}')
        return found
    def needs(self):
        "Every environment key the pipeline names, in the order the steps name them."
        return list(L(self.steps).attrgot('needs').concat().unique())
    def missing(self, step):
        "Keys this step names that the environment store cannot produce."
        try: return [k for k in step.needs if not self.env.get(k)]
        except Exception: return [k for k in step.needs if not os.environ.get(k)]
    def _env(self):
        "The child's environment: this process, the project's interpreter, then the store on top."
        env = venv_env(self.python)
        try: env.update({k: str(v) for k, v in self.env.values(self.needs()).items()})
        except Exception: pass
        env.setdefault('PYTHONUNBUFFERED', '1')
        return env
    def _row(self, s):
        "One step as a caller sees it: the plan, plus how the last run of it went."
        r = self.results.get(s.id) or {}
        return s.dict() | {
            'status': 'running' if s.id == self.active else (r.get('status') or 'pending'),
            'code': r.get('code'), 'started': r.get('started', 0), 'finished': r.get('finished', 0),
            'missing': self.missing(s)}
    def state(self):
        "Everything about this pipeline that a caller could want to draw or assert on."
        rows = [self._row(s) for s in self.steps]
        failed = first(rows, lambda r: r['status'] == 'failed')
        return {'root': str(self.root), 'kind': self.KIND, 'path': str(self.path),
            'configured': self.path.exists(), 'steps': rows, 'active': self.active,
            'running': bool(self.active), 'auto': self._auto,
            'failed': failed['id'] if failed else '',
            'done': bool(rows) and all(r['status'] in ('passed', 'skipped') for r in rows),
            'needs': self.needs()}
    def subscribe(self):
        "A queue of terminal frames for one viewer, primed with what has already scrolled by."
        q = asyncio.Queue()
        if self.log: q.put_nowait(bytes(self.log))
        self._subs.add(q)
        return q
    def unsubscribe(self, q): self._subs.discard(q)
    def _broadcast(self, data):
        self.log += data
        if len(self.log) > SCROLLBACK: del self.log[:-SCROLLBACK]
        for q in list(self._subs):
            try: q.put_nowait(data)
            except Exception: self._subs.discard(q)
    def write(self, data):
        "Keystrokes into the running step: twine asks for a password."
        if self.pty is not None and self.pty.alive: self.pty.write(data)
    def resize(self, cols, rows):
        if self.pty is not None: self.pty.resize(rows, cols)
    def tail(self, lines=80):
        "The end of the transcript as plain text, escapes gone and bare returns made newlines."
        text = strip_ansi(bytes(self.log).decode('utf-8', 'replace'))
        rows = text.replace('\r\n', '\n').replace('\r', '\n').rstrip('\n').split('\n')
        return '\n'.join(rows[-max(1, int(lines)):])
    def _resolve(self, argv, env):
        "The executable to spawn, from this pipeline's interpreter."
        return uv_argv(self.root, argv) or resolve_script(argv, self.python, env)
    async def start(self, step_id='', auto=True):
        "Run one step, then keep going while they pass. Refuses to start a second one."
        if self.active: raise PipelineError(f'{self.active} is still running')
        self._auto = bool(auto)
        step = self.step(step_id) if step_id else self._next()
        if step is None: raise PipelineError('every step has already run — reset to run them again')
        await self._run(step)
        return self.state()
    def _next(self):
        "The first step that has not passed or been skipped."
        done = ('passed', 'skipped')
        return first(self.steps, lambda s: (self.results.get(s.id) or {}).get('status') not in done)
    def _after(self, step):
        i = [s.id for s in self.steps].index(step.id)
        return self.steps[i + 1] if i + 1 < len(self.steps) else None
    async def _run(self, step):
        from ptymini.core import PtySession
        argv = list(step.argv) if step.argv else shlex.split(step.cmd)
        if not argv: raise PipelineError(f'{step.id} has no command to run')
        self.active = step.id
        self.results[step.id] = {'status': 'running', 'code': None,
            'started': int(time.time()), 'finished': 0}
        env = self._env()
        self._broadcast(f'\r\n\x1b[1;36m❯ {step.cmd}\x1b[0m\r\n'.encode())
        try:
            self.pty = PtySession(self._resolve(argv, env), cwd=str(self.root), env=env,
                                  rows=30, cols=100, buffer_bytes=SCROLLBACK)
        except OSError as e:
            self.active = ''
            self.results[step.id] = {'status': 'failed', 'code': None, 'started': int(time.time()),
                'finished': int(time.time()), 'error': str(e)}
            self._broadcast(f'\r\n\x1b[31m{step.cmd}: {e}\x1b[0m\r\n'.encode())
            return
        self._pump = asyncio.create_task(self._drain(step))
    async def _drain(self, step):
        "Pump the step's pty to every viewer, record how it ended, then decide what happens next."
        pty = self.pty
        # from_start, because the child can write and exit before this task first runs. The
        # ring is this step's own, so replaying it from its first byte duplicates nothing.
        try:
            async for chunk in pty.attach(from_start=True): self._broadcast(chunk)
        except Exception as e:
            self._broadcast(f'\r\n\x1b[31m{type(e).__name__}: {e}\x1b[0m\r\n'.encode())
        code = pty.exit_code
        ok = code == 0
        self.results[step.id] = {'status': 'passed' if ok else 'failed', 'code': code,
            'started': (self.results.get(step.id) or {}).get('started', 0),
            'finished': int(time.time())}
        self.active = ''
        note = '\x1b[32m✓ ' if ok else '\x1b[31m✗ '
        self._broadcast(f'\r\n{note}{step.label} exited {code}\x1b[0m\r\n'.encode())
        if ok and self._auto and (nxt := self._after(step)) is not None: await self._run(nxt)
    def stop(self):
        "Stop the pipeline where it is. The step is killed; what it already did stays done."
        import signal
        self._auto = False
        step_id, self.active = self.active, ''
        if self.pty is not None and self.pty.alive:
            try: self.pty.kill(signal.SIGTERM)
            except ProcessLookupError: pass
        if step_id:
            self.results[step_id] = (self.results.get(step_id) or {}) | {
                'status': 'failed', 'code': None, 'finished': int(time.time())}
            self._broadcast(f'\r\n\x1b[33m{step_id} stopped\x1b[0m\r\n'.encode())
        return self.state()
    def skip(self, step_id):
        "Mark a step done without running it."
        step = self.step(step_id)
        if self.active == step.id: raise PipelineError(f'{step.id} is still running')
        self.results[step.id] = {'status': 'skipped', 'code': None, 'started': 0,
                                 'finished': int(time.time())}
        return self.state()
    def reset(self):
        "Forget what ran. The commands are unchanged."
        if self.active: raise PipelineError(f'{self.active} is still running')
        self.results.clear()
        self.log.clear()
        self._broadcast(b'\x1b[2J\x1b[H')
        return self.state()
    def blame(self, checkouts=(), family=None):
        "Which of your own packages raised in the last failure, pointed at your checkout of it."
        from pullup.stack import blame as attribute
        if not any((self.results.get(s.id) or {}).get('status') == 'failed' for s in self.steps):
            return {}
        return attribute(self.tail(400), checkouts=checkouts, family=family)

The plan is written at `root/dir/FILE` and nothing is written anywhere else. A missing file, an unreadable one, or one with no runnable command in it gives the class's `defaults` instead of an error. A half-written plan is a reason to offer the default, not to make a panel unopenable.

Reading normalises what it finds. A row with no `cmd` is dropped, and a row with no `id` is named by its position. `save` writes the rows it is given and reads them back, so what a caller holds afterwards is what the file says. It refuses a plan with no runnable command left in it.

In [11]:
#| hide
demo = TemporaryDirectory(); proj = Path(demo.name)

In [12]:
p = Pipeline(proj, dir='.demo')
st = p.save([{'label': 'say hello', 'cmd': 'echo hello from a pipeline'},
             {'id': 'ok', 'label': 'exit clean', 'cmd': 'true'},
             {'label': 'no command of its own'}])
[(r['id'], r['label'], r['status']) for r in st['steps']]

[('step1', 'say hello', 'pending'), ('ok', 'exit clean', 'pending')]

In [13]:
#| hide
test_eq([s.id for s in Pipeline(proj, dir='.demo').steps], ['step1', 'ok'])
test_fail(lambda: p.save([{'label': 'nothing to run'}]), contains='at least one step')
test_eq(Pipeline(proj).steps, [])          # the base class has no default plan
assert not (proj/'.pullup').exists(), 'nothing is written outside the directory the caller named'

`start` runs one step. While `auto`, it runs the next one each time a step passes. It stops at the first failure. A release that publishes from a tree whose tests never passed is worse than one that stops.

`start` raises while a step is running. Two children on one terminal would interleave their output and neither could be answered.

The transcript records the command before it runs and the exit code after, so a reader can tell which output came from which step.

In [14]:
#| hide
async def settle(p, secs=30):
    "Wait until nothing is running, so the next line can show the finished state."
    for _ in range(int(secs*20)):
        if not p.state()['running']: return p.state()
        await asyncio.sleep(.05)
    raise RuntimeError('still running')

In [15]:
await p.start()
st = await settle(p)
[(r['id'], r['status'], r['code']) for r in st['steps']]

[('step1', 'passed', 0), ('ok', 'passed', 0)]

In [16]:
print(p.tail(9))


❯ echo hello from a pipeline
hello from a pipeline

✓ say hello exited 0

❯ true

✓ exit clean exited 0


In [17]:
#| hide
test_eq((st['done'], st['failed'], st['running']), (True, '', False))
assert 'hello from a pipeline' in p.tail(), 'the step output itself is in the transcript'
test_fail(lambda: p.step('nope'), contains='unknown pipeline step: nope')

A failing step leaves every step after it `pending`. The exit code is the child's own.

In [18]:
q = Pipeline(proj, dir='.fail')
q.save([{'id': 'tests', 'label': 'tests', 'cmd': 'sh -c "exit 3"'},
        {'id': 'publish', 'label': 'publish', 'cmd': 'echo publishing'}])
await q.start()
[(r['id'], r['status'], r['code']) for r in (await settle(q))['steps']]

[('tests', 'failed', 3), ('publish', 'pending', None)]

In [19]:
#| hide
test_eq(q.state()['failed'], 'tests')
assert 'publishing' not in q.tail(), 'the step after a failure never ran'

`subscribe` answers with an `asyncio.Queue` of terminal frames, primed with everything that has already scrolled by. A viewer that joins in the middle of a release sees the run from its first byte. `unsubscribe` drops the queue.

The transcript is kept to the last `SCROLLBACK` bytes. `tail` decodes it, strips the escape sequences, and turns bare carriage returns into newlines, so what a progress bar drew over itself reads as the lines it drew.

In [20]:
sub = p.subscribe()
b'hello from a pipeline' in sub.get_nowait()

True

In [21]:
#| hide
p.unsubscribe(sub)
test_eq(Pipeline(proj, dir='.demo').subscribe().qsize(), 0)   # nothing has scrolled by yet
assert '\x1b' not in p.tail() and '\r' not in p.tail()

`needs` is every environment key the plan names, each once, in the order the steps name them. `missing` is the keys one step names that the store cannot produce. A panel greys out a step from that list before anyone presses it.

Neither hands a value back. `missing` asks the store only whether there is one. The values go into the child's environment when the step runs, so a command reads `TWINE_PASSWORD` without it ever appearing on a command line.

`env` is an `EnvStore`, or anything with `get(key)` and `values(keys)`.

In [22]:
#| hide
class Store:
    "An environment store holding exactly what the example says it holds."
    def __init__(self, **kw): self.kw = kw
    def get(self, key, secret=True): return self.kw.get(key, '')
    def values(self, keys, secret=True): return {k: self.kw[k] for k in keys if self.kw.get(k)}

In [23]:
r = Pipeline(proj, dir='.needs', env=Store(GITHUB_TOKEN='ghp_pretend'))
r.save([{'id': 'gh', 'label': 'GitHub release', 'cmd': 'true', 'needs': ['GITHUB_TOKEN']},
        {'id': 'pypi', 'label': 'PyPI', 'cmd': 'true', 'needs': ['TWINE_PASSWORD', 'GITHUB_TOKEN']}])
r.needs(), [(s.id, r.missing(s)) for s in r.steps]

(['GITHUB_TOKEN', 'TWINE_PASSWORD'],
 [('gh', []), ('pypi', ['TWINE_PASSWORD'])])

`skip` marks a step done without running it, and it counts as done. Skipping is a decision, not a gap.

`reset` forgets every result and clears the transcript. The commands are unchanged. `reset_plan` puts the defaults back without deleting the file, so `configured` still reports a plan on disk and the next read finds it again.

`stop` kills the running step and turns auto-advance off. What already passed stays passed. The stopped step is `failed` with no exit code, because nothing else can be said about a process that was killed.

`blame` reads the last 400 lines of the transcript for the deepest frame in one of your own packages. It answers `{}` where no step has failed.

In [24]:
#| hide
test_eq(p.blame(), {})                                        # nothing failed, nothing to attribute
test_eq([x['status'] for x in p.reset()['steps']], ['pending', 'pending'])
test_eq(p.tail(), '')                                         # the transcript went with the results
p.skip('ok')
test_eq(p.state()['steps'][1]['status'], 'skipped')
plain = p.reset_plan()
test_eq(plain['steps'], [])
assert plain['configured'] and p.path.exists(), 'the file stayed; only the plan in hand changed'
test_eq([s.id for s in Pipeline(proj, dir='.demo').steps], ['step1', 'ok'])

In [25]:
#| export
class Release(Pipeline):
    "The release pipeline: nbdev's commands, fastship's for a package that is not one, else plain."
    FILE = 'release.json'
    KIND = 'release'
    @staticmethod
    def defaults(root): return default_steps(root)
    def state(self):
        flow = release_flow(self.root)
        return super().state() | {'flow': flow, 'nbdev': flow == 'nbdev',
            'fastship': flow == 'fastship', 'app': app_project(self.root)}

`Release` is `Pipeline` with the plan for whatever kind of project the folder holds: nbdev's commands for an nbdev repo, fastship's for a Python package that is not one, cargo's for a crate, and a plain build and upload otherwise. `pullup.project` decides which, and `state` reports it as `flow`.

The plan is `release.json`, beside `pipeline.json` rather than over it, so one project can hold both.

`app` is true where the project also packages itself into a desktop bundle. That adds the bundle and its upload after the last step that publishes.

In [26]:
#| hide
rel = TemporaryDirectory(); rroot = Path(rel.name)
(rroot/'settings.ini').write_text('[DEFAULT]\nlib_name = demo\nnbs_path = nbs\n')

41

In [27]:
rl = Release(rroot)
rl.state()['flow'], [s.id for s in rl.steps], rl.path.name

('nbdev', ['prepare', 'bump', 'gh', 'pypi'], 'release.json')

In [28]:
#| hide
test_eq((rl.state()['nbdev'], rl.state()['fastship'], rl.state()['app']), (True, False, False))
(rroot/'settings.ini').unlink()
(rroot/'pyproject.toml').write_text('[project]\nname = "demo"\n')
test_eq(Release(rroot).state()['flow'], 'fastship')
for t in (tmp, demo, rel): t.cleanup()